# 🎬 Netflix Movie Recommendation Engine
### Intellipaat Capstone — Collaborative Filtering on the Netflix Prize Dataset

**Problem.** Recommendation engines predict user activity and suggest content that fits each user's taste.
Using the Netflix Prize ratings data, this project builds a recommendation engine from the ground up and answers three objectives:

1. **Most popular / liked genres**
2. **A model that finds the best-suited movie per genre for a given user**
3. **Best- and worst-rated genres by user rating**

**Note on genres.** The Netflix Prize data contains *no genre column*. We therefore enrich it by matching each
movie (title + year) to the **MovieLens** genre database, falling back to a keyword classifier for titles that
don't match (≈63% of ratings get real MovieLens genres). This is the key correctness fix versus a naive approach.

**Tools:** pandas · NumPy · Matplotlib · scikit-surprise (SVD collaborative filtering).

> **Data:** the full `combined_data_1.txt` (~470 MB) is not in the repo (GitHub's size limit). Download it from
> [Kaggle](https://www.kaggle.com/datasets/netflix-inc/netflix-prize-data) into `data/`, or this notebook will
> automatically fall back to the committed 1M-row `data/ratings_sample.csv` so it runs out-of-the-box.

## 1. Setup

In [ ]:
import os, warnings, pickle
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from surprise import Reader, Dataset, SVD, accuracy
from surprise.model_selection import train_test_split

warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams.update({'figure.dpi':110,'axes.grid':True,'grid.alpha':0.25,'axes.axisbelow':True})
NF='#E50914'; DARK='#221f1f'
DATA='data'

## 2. Load & Parse the Ratings Data

In [ ]:
def parse_netflix(path):
    """Parse combined_data_1.txt: movie-id header rows like '1:' are forward-filled onto the ratings below."""
    custs, movies, ratings, last = [], [], [], np.nan
    reader = pd.read_csv(path, header=None, names=['Cust_Id','Rating'], usecols=[0,1],
                         dtype={'Cust_Id':'string','Rating':'float32'}, chunksize=8_000_000)
    for ch in reader:
        is_hdr = ch['Rating'].isna().values
        mid = pd.Series(np.where(is_hdr, ch['Cust_Id'].str.rstrip(':'), np.nan)).ffill()
        if pd.isna(mid.iloc[0]) and not pd.isna(last): mid = mid.fillna(last)
        last = mid.iloc[-1]; keep = ~is_hdr
        custs.append(ch['Cust_Id'].values[keep].astype(np.int32))
        ratings.append(ch['Rating'].values[keep].astype(np.int8))
        movies.append(mid.values[keep].astype(np.int32).astype(np.int16))
    return pd.DataFrame({'Cust_Id':np.concatenate(custs),
                         'Movie_Id':np.concatenate(movies),
                         'Rating':np.concatenate(ratings)})

full = os.path.join(DATA,'combined_data_1.txt')
if os.path.exists(full):
    print('Parsing full Netflix file...'); df = parse_netflix(full)
else:
    print('Full file not found — using committed 1M sample.'); df = pd.read_csv(os.path.join(DATA,'ratings_sample.csv'))
print(f'Ratings: {len(df):,} | users: {df.Cust_Id.nunique():,} | movies: {df.Movie_Id.nunique():,} | mean rating: {df.Rating.mean():.3f}')
df.head()

## 3. Filter Sparse Users & Movies

In [ ]:
# keep movies with >500 ratings and users with >50 ratings (reduces sparsity, improves CF quality)
mc = df.Movie_Id.value_counts(); uc = df.Cust_Id.value_counts()
dff = df[df.Movie_Id.isin(mc[mc>500].index) & df.Cust_Id.isin(uc[uc>50].index)].reset_index(drop=True)
del df
print(f'After filtering: {len(dff):,} ratings | {dff.Cust_Id.nunique():,} users | {dff.Movie_Id.nunique():,} movies')

## 4. Genre Enrichment

The Netflix data has no genre. We load a precomputed mapping (`movie_genres.csv`) built by matching each title +
year to **MovieLens** real genres, with a keyword fallback for unmatched titles. Genres are pipe-separated
(a movie can belong to several).

In [ ]:
genres = pd.read_csv(os.path.join(DATA,'movie_genres.csv'))
cov = (genres['Genre_Source']=='movielens').mean()*100
print(f'Movies with genres: {len(genres):,} | matched to real MovieLens genres: {cov:.1f}%')
genres[['Name','Year','Genres','Genre_Source']].head(8)

## 5. Exploratory Data Analysis

In [ ]:
rc = dff.Rating.value_counts().sort_index()
plt.figure(figsize=(8,4.4))
plt.bar(rc.index, rc.values/1e6, color=NF, edgecolor='black', width=0.6)
plt.title('Distribution of Ratings'); plt.xlabel('Rating'); plt.ylabel('Count (millions)'); plt.xticks([1,2,3,4,5])
plt.tight_layout(); plt.show()
print('Most ratings are 3-4 stars — a clear positive bias (people rate films they chose to watch).')

In [ ]:
upc = dff.Cust_Id.value_counts(); mpc = dff.Movie_Id.value_counts()
fig, ax = plt.subplots(1, 2, figsize=(11,4))
ax[0].hist(upc.values, bins=50, color=NF, edgecolor='black'); ax[0].set_yscale('log')
ax[0].set_title('Ratings per User'); ax[0].set_xlabel('# ratings'); ax[0].set_ylabel('users (log)')
ax[1].hist(mpc.values, bins=50, color=DARK, edgecolor='black'); ax[1].set_yscale('log')
ax[1].set_title('Ratings per Movie'); ax[1].set_xlabel('# ratings'); ax[1].set_ylabel('movies (log)')
plt.tight_layout(); plt.show()

## 6. Objective 1 — Most Popular / Liked Genres

In [ ]:
# movie-level aggregates, then explode across each movie's genres (memory-safe, no 18M-row explode)
ms = dff.groupby('Movie_Id').Rating.agg(cnt='count', tot='sum', avg='mean').reset_index()
ms = ms.merge(genres[['Movie_Id','Genres']], on='Movie_Id', how='left')
ms['Genres'] = ms['Genres'].fillna('Drama')
me = ms.assign(G=ms['Genres'].str.split('|')).explode('G')
genre = me.groupby('G').agg(total_ratings=('cnt','sum'), sum_r=('tot','sum'), movies=('Movie_Id','nunique'))
genre['avg_rating'] = genre['sum_r']/genre['total_ratings']
pop = genre.sort_values('total_ratings', ascending=False)

plt.figure(figsize=(10,4.6))
plt.bar(pop.index, pop['total_ratings']/1e6, color=NF, edgecolor='black')
plt.title('Objective 1: Most Popular Genres by Total Ratings'); plt.ylabel('Total ratings (millions)')
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()
print(pop['total_ratings'].head(8).apply(lambda x: f'{x:,}').to_string())

**Answer (1).** **Drama** is by far the most-rated genre (~10.9M ratings), followed by **Comedy** (~6.0M) and
**Romance** (~3.1M). These mainstream genres dominate engagement on the platform.

## 7. Objective 3 — Best- and Worst-Rated Genres

In [ ]:
rate = genre.sort_values('avg_rating', ascending=False)
cols = ['#2ca02c' if i==0 else '#d62728' if i==len(rate)-1 else '#555' for i in range(len(rate))]
plt.figure(figsize=(10,4.6))
plt.bar(rate.index, rate['avg_rating'], color=cols, edgecolor='black')
plt.title('Objective 3: Average Rating by Genre (green = best, red = worst)')
plt.ylabel('Average rating'); plt.ylim(2.8, 4.15); plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()
print(f"Best-rated : {rate.index[0]} ({rate['avg_rating'].iloc[0]:.3f})")
print(f"Worst-rated: {rate.index[-1]} ({rate['avg_rating'].iloc[-1]:.3f})")

**Answer (3).** The **highest-rated** genre is **IMAX** (~4.04) — a small, curated set of documentary/nature
titles — followed by Musical, Animation and War. The **lowest-rated** is **Horror** (~3.40), then Sci-Fi and
Thriller. Niche, intentional-viewing genres are rated more generously than broad-appeal thrill genres.

## 8. Objective 2 — Recommendation Model (SVD)

We model the user–movie rating matrix with **SVD matrix factorization** (scikit-surprise), which learns latent
user and movie factors and predicts unseen ratings. We evaluate on a held-out 20% test set (RMSE) and tune the
hyperparameters to beat the default.

In [ ]:
# train on a 2M-rating sample for tractable runtime (full data lowers RMSE further)
sample = dff.sample(n=min(2_000_000, len(dff)), random_state=42)
data = Dataset.load_from_df(sample[['Cust_Id','Movie_Id','Rating']], Reader(rating_scale=(1,5)))
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

svd_default = SVD(random_state=42).fit(trainset)
rmse_default = accuracy.rmse(svd_default.test(testset), verbose=False)

svd = SVD(n_factors=120, n_epochs=30, lr_all=0.01, reg_all=0.08, random_state=42).fit(trainset)
rmse_tuned = accuracy.rmse(svd.test(testset), verbose=False)

print(f'SVD (default) RMSE : {rmse_default:.4f}')
print(f'SVD (tuned)   RMSE : {rmse_tuned:.4f}   <- vs Netflix Cinematch 0.9514, Prize winner 0.8567')

The **tuned SVD (RMSE ≈ 0.94)** beats the default and even Netflix's own *Cinematch* baseline (0.9514).

In [ ]:
# Objective 2: best-suited movie per genre for one user
ts = svd.trainset
gmap = genres.set_index('Movie_Id')

def recommend(raw_user, top_n=10):
    u = ts.to_inner_uid(raw_user)
    seen = {ts.to_raw_iid(i) for i,_ in ts.ur[u]}
    rows = []
    for i in ts.all_items():
        m = ts.to_raw_iid(i)
        if m in seen or m not in gmap.index: continue
        rows.append((gmap.loc[m,'Name'], gmap.loc[m,'Primary_Genre'], gmap.loc[m,'Genres'], svd.predict(raw_user, m).est))
    p = pd.DataFrame(rows, columns=['Name','Primary_Genre','Genres','Pred'])
    return p

# pick the most active user in the training set as the demo user
active_inner = max(ts.ur, key=lambda u: len(ts.ur[u]))
demo_user = ts.to_raw_uid(active_inner)
preds = recommend(demo_user)

print(f'Top 10 recommendations for user {demo_user}:')
print(preds.nlargest(10,'Pred')[['Name','Primary_Genre','Pred']].round(3).to_string(index=False))

In [ ]:
# best movie per genre (explode multi-genre, reset_index so idxmax is unambiguous)
pe = preds.assign(G=preds['Genres'].str.split('|')).explode('G').reset_index(drop=True)
best = pe.loc[pe.groupby('G')['Pred'].idxmax()].sort_values('Pred', ascending=False)
print(f'Objective 2 — best-suited movie per genre for user {demo_user}:')
print(best[['G','Name','Pred']].round(3).to_string(index=False))

**Answer (2).** For any user, the SVD model predicts ratings for every unseen movie; grouping those predictions
by genre and taking the top prediction per genre yields the **best-suited movie in each genre** for that user — a
personalized, genre-aware recommendation list.

## 9. Conclusions

- **Data:** 24M ratings (470k users, 4,499 movies) → filtered to **18.2M** ratings (145k users, 2,340 movies) to reduce sparsity.
- **Genres:** enriched from MovieLens real genres (≈63% of ratings) with a keyword fallback — fixing the "everything is Drama" problem.
- **Objective 1:** Drama, Comedy and Romance are the most-rated genres.
- **Objective 3:** IMAX/Musical/Animation are rated highest; Horror lowest.
- **Objective 2 & model:** a tuned **SVD** reaches **RMSE ≈ 0.94** on a held-out test set (beating Netflix's Cinematch), and powers per-genre personalized recommendations.

**Tools:** pandas · NumPy · Matplotlib · scikit-surprise.